In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import json, os, re, time, subprocess, ast
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

INPUT_DIR = '/kaggle/input/datasets/apr-dataset'
OUTPUT_DIR = '/kaggle/working'

KB_PATH = f'{INPUT_DIR}/pattern_knowledge_base_final.json'
TEST_PATH = f'{INPUT_DIR}/test_loop_filtered_verified.jsonl'
TEST_CASES_DIR = f'{INPUT_DIR}/public_test_cases/codenet/public_test_cases'
OUTPUT_PATH = f'{OUTPUT_DIR}/phase2_batch_results_gpt54mini_k10.jsonl'

SPEEDUP_THRESHOLD = 1.10
MAX_ATTEMPTS = 10
GPT_MODEL = "gpt-5.4-mini"  # verify exact string in OpenAI's docs before running

user_secrets = UserSecretsClient()
client = OpenAI(api_key=user_secrets.get_secret("OPENAI_API_KEY"))

with open(KB_PATH) as f:
    pattern_knowledge_base = json.load(f)
print(f'Loaded KB: {len(pattern_knowledge_base)} patterns')

test_samples = []
with open(TEST_PATH) as f:
    for line in f:
        test_samples.append(json.loads(line))
print(f'Loaded {len(test_samples)} test samples')

print('Test cases dir exists:', os.path.isdir(TEST_CASES_DIR))

In [ ]:
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
import time

GPT_MODEL = "gpt-5.4-mini"  # verify exact model string in OpenAI's docs before running

user_secrets = UserSecretsClient()
client = OpenAI(api_key=user_secrets.get_secret("OPENAI_API_KEY"))

def call_gen_llm(prompt, max_new_tokens=1024, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=GPT_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.15,
                max_completion_tokens=max_new_tokens,
            )
            content = response.choices[0].message.content
            return content.strip() if content else ''
        except Exception as e:
            err_str = str(e).lower()
            if 'rate_limit' in err_str or '429' in err_str:
                wait_time = 15 * (attempt + 1)
                print(f'   Rate limited, waiting {wait_time}s...')
                time.sleep(wait_time)
            else:
                print(f'   API error: {e}')
                return ''
    return ''

print(f'{GPT_MODEL} ready (temperature=0.3, max_new_tokens=1024).')

In [ ]:
def extract_code_from_response(response):
    match = re.search(r"```(?:python)?\s*\n(.*?)```", response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return response.strip()

print('Code extractor ready.')


In [ ]:
def load_test_cases(problem_id, test_cases_dir):
    problem_dir = os.path.join(test_cases_dir, problem_id)
    if not os.path.isdir(problem_dir):
        return []
    cases = []
    i = 0
    while True:
        input_path = os.path.join(problem_dir, f'input.{i}.txt')
        output_path = os.path.join(problem_dir, f'output.{i}.txt')
        if not os.path.exists(input_path):
            break
        with open(input_path) as f:
            stdin_data = f.read()
        with open(output_path) as f:
            expected_output = f.read().strip()
        cases.append({'stdin': stdin_data, 'expected': expected_output})
        i += 1
    return cases

def check_compiles(code):
    try:
        compile(code, '<repair>', 'exec')
        return True, None
    except SyntaxError as e:
        return False, str(e)

def run_with_timeout(code, stdin_data, timeout_sec=5):
    start = time.time()
    try:
        result = subprocess.run(['python3', '-c', code], input=stdin_data,
                                 capture_output=True, text=True, timeout=timeout_sec)
        elapsed = time.time() - start
        return result.stdout.strip(), result.returncode == 0, elapsed
    except subprocess.TimeoutExpired:
        return None, False, timeout_sec

def check_functional_correctness(code, test_cases):
    for case in test_cases:
        output, success, _ = run_with_timeout(code, case['stdin'])
        if not success or output != case['expected']:
            return False, {**case, 'actual_output': output if output is not None else '(crashed or timed out)'}
    return True, None

def measure_runtime(code, test_cases, repeats=3):
    total_time = 0
    for _ in range(repeats):
        for case in test_cases:
            _, success, elapsed = run_with_timeout(code, case['stdin'])
            if not success:
                return None
            total_time += elapsed
    return total_time / repeats

def validate_repair(slow_code, repaired_code, problem_id, test_cases_dir, speedup_threshold=SPEEDUP_THRESHOLD):
    test_cases = load_test_cases(problem_id, test_cases_dir)
    if not test_cases:
        return {'verdict': 'RETRY', 'reason': f'No test cases found for {problem_id}'}
    compiles, syntax_error = check_compiles(repaired_code)
    if not compiles:
        return {'verdict': 'RETRY', 'reason': 'compile_error', 'detail': syntax_error}
    correct, failing_case = check_functional_correctness(repaired_code, test_cases)
    if not correct:
        return {'verdict': 'RETRY', 'reason': 'test_failure', 'detail': failing_case}
    slow_runtime = measure_runtime(slow_code, test_cases)
    repaired_runtime = measure_runtime(repaired_code, test_cases)
    if slow_runtime is None or repaired_runtime is None or repaired_runtime <= 0:
        return {'verdict': 'RETRY', 'reason': 'runtime_measurement_failed'}
    speedup = slow_runtime / repaired_runtime
    if speedup >= speedup_threshold:
        return {'verdict': 'ACCEPT', 'speedup': round(speedup, 3),
                'slow_runtime': round(slow_runtime, 4), 'repaired_runtime': round(repaired_runtime, 4)}
    else:
        return {'verdict': 'RETRY', 'reason': 'insufficient_speedup', 'speedup': round(speedup, 3),
                'slow_runtime': round(slow_runtime, 4), 'repaired_runtime': round(repaired_runtime, 4)}

def load_completed_problem_ids(output_path):
    completed = set()
    if os.path.exists(output_path):
        with open(output_path) as f:
            for line in f:
                try:
                    completed.add(json.loads(line)['problem_id'])
                except Exception:
                    continue
    return completed

print('Validation + checkpoint helpers ready.')


In [ ]:
def build_cot_init_prompt(slow_code):
    return f"""You are an expert Python engineer specializing in performance optimization.

First, think step by step: identify the performance bottleneck in the code below.
Explain your reasoning briefly as comments, then apply an initial optimization.

Slow code:
{slow_code}

Return your brief reasoning as comments, followed by the optimized Python code.
Wrap the final code in a single ```python code block."""


def build_refinement_prompt(slow_code, initial_code):
    return f"""You previously produced this candidate optimization:

{initial_code}

Now refine this candidate by applying additional transformation patterns commonly
used for performance optimization: loop restructuring, reducing redundant
computation, using built-in functions (sum, sorted, collections.Counter) instead
of manual loops, eliminating unnecessary nested loops, and using more efficient
data structures (sets/dicts for membership tests) where applicable.

Original slow code (for reference):
{slow_code}

Return ONLY the refined, final optimized Python code. No markdown. No explanations."""


def sbllm_repair(slow_code, problem_id, test_cases_dir):
    init_prompt = build_cot_init_prompt(slow_code)
    initial_code = extract_code_from_response(call_gen_llm(init_prompt))

    refine_prompt = build_refinement_prompt(slow_code, initial_code)
    refined_code = extract_code_from_response(call_gen_llm(refine_prompt))

    result = validate_repair(slow_code, refined_code, problem_id, test_cases_dir)
    result['attempt'] = 1
    return {
        'final_verdict': result['verdict'] if result['verdict'] == 'ACCEPT' else 'RETRY_EXHAUSTED',
        'code': refined_code,
        'initial_code': initial_code,
        'history': [result]
    }

print('SBLLM (single-iteration variant) repair function ready.')


In [ ]:
SBLLM_OUTPUT = f'{OUTPUT_DIR}/sbllm_baseline_gpt54mini.jsonl'
print(f'GPT-5.4-mini SBLLM output will be saved to: {SBLLM_OUTPUT}')

In [ ]:
completed = load_completed_problem_ids(SBLLM_OUTPUT)
print(f'Already completed: {len(completed)} / {len(test_samples)}')

with open(SBLLM_OUTPUT, 'a') as out_f:
    for i, sample in enumerate(test_samples):
        problem_id = sample.get('problem_id')
        if problem_id in completed:
            continue
        slow_code = sample['input']
        print(f'[{i+1}/{len(test_samples)}] {problem_id}...')
        start = time.time()
        try:
            result = sbllm_repair(slow_code, problem_id, TEST_CASES_DIR)
            record = {'problem_id': problem_id, 'final_verdict': result['final_verdict'],
                      'num_attempts': 1, 'history': result['history'],
                      'final_code': result['code'], 'elapsed_seconds': round(time.time() - start, 2)}
        except Exception as e:
            record = {'problem_id': problem_id, 'final_verdict': 'PIPELINE_ERROR', 'error': str(e)}
        out_f.write(json.dumps(record) + '\n')
        out_f.flush()

print('SBLLM (GPT-5.4-mini) run complete.')


In [ ]:
import statistics

def score_results(path, label):
    results = []
    with open(path) as f:
        for line in f:
            results.append(json.loads(line))

    total = len(results)
    accepted = [r for r in results if r.get('final_verdict') == 'ACCEPT']
    pass1_count = sum(
        1 for r in results
        if r.get('history') and (
            r['history'][0].get('verdict') == 'ACCEPT' or
            r['history'][0].get('reason') == 'insufficient_speedup'
        )
    )

    print(f'=== {label} ===')
    print(f'Total: {total}')
    print(f'%OPT: {100*len(accepted)/total:.1f}%')
    print(f'Pass@1: {100*pass1_count/total:.1f}%')

    speedups = [r['history'][-1]['speedup'] for r in accepted if 'speedup' in r['history'][-1]]
    if speedups:
        print(f'Median speedup: {statistics.median(speedups):.3f}x')
        print(f'Mean speedup: {statistics.mean(speedups):.3f}x')

score_results(SBLLM_OUTPUT, 'SBLLM (GPT-5.4-mini, single-iteration)')